# MedGemma 4B-PT. Fine-tuning QLoRA sur RSNA

**Setup obligatoire :** `Exécution > Modifier le type d'exécution > T4 GPU`

**Pipeline :**
1. Upload `rsna_samples.csv` + `kaggle.json`
2. Téléchargement dataset Kaggle
3. Construction du dataset conversation (image + prompt → réponse label)
4. Chargement `google/medgemma-4b-pt` en 4-bit + adaptateurs QLoRA (PEFT)
5. Fine-tuning avec `trl.SFTTrainer`
6. Évaluation sur le split val
7. Sauvegarde & téléchargement des poids LoRA

> > /!\ Prototype pédagogique. Non destiné au diagnostic médical. Toute utilisation clinique requiert validation par un professionnel qualifié.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 0. Variables d'environnement & utilitaires de reproductibilité
# ═══════════════════════════════════════════════════════════════════
# Doit être exécutée EN PREMIER, avant tout import PyTorch/CUDA.
# PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True réduit la fragmentation
# mémoire GPU, utile pour les grands modèles chargés en 4-bit.

import os, random

# Reproductibilité globale
# toutes les cellules suivantes importent la variable SEED plutôt que de coder 42 en dur partout
SEED = 42

def set_all_seeds(seed: int = SEED) -> None:
    # Fixe les seeds Python, NumPy et PyTorch (CPU + GPU)
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    try:
        import numpy as np
        np.random.seed(seed)
    except ImportError:
        pass
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        # Déterminisme CUDA, légèrement plus lent mais résultats identiques
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark     = False
    except ImportError:
        pass

set_all_seeds(SEED)

# Réduit fragmentation mémoire GPU
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

print(f'Seed globale fixée à {SEED}')
print('Variables CUDA configurées.')


Seed globale fixée à 42
Variables CUDA configurées.


In [ ]:
# ════════════════════════════════════
# STEP 1. Installation des dépendances
# ════════════════════════════════════
# Packages nécessaires :
#   transformers ≥ 4.50  → supporte MedGemma / Gemma-3 architecture
#   accelerate           → dispatching multi-GPU et mixed-precision
#   bitsandbytes         → quantification 4-bit NF4 (QLoRA)
#   peft                 → LoRA / QLoRA (adaptateurs légers)
#   trl                  → SFTTrainer (fine-tuning supervisé simplifié)
#   pillow               → chargement et conversion des images DICOM/PNG
#   scikit-learn         → métriques d'évaluation (accuracy, F1, rapport)
#   datasets             → format Dataset HuggingFace pour le trainer

!pip install -q transformers>=4.50.0 accelerate bitsandbytes peft trl pillow scikit-learn datasets
print('Installation OK')


Installation OK


In [ ]:
# ════════════════════════════════════════
# STEP 2. Upload et chargement du CSV RSNA
# ════════════════════════════════════════
# Le fichier rsna_samples.csv doit contenir au minimum les colonnes :
#   - case_id : identifiant unique du cas
#   - label   : classe cible parmi {normal, suspected_opacity, uncertain}
# Les colonnes 'split' et 'quality' sont supprimées si présentes (le split sera recalculé de manière reproductible en Cellule 5)

from google.colab import files
import io, os
import pandas as pd

print('Uploadez rsna_samples.csv')
uploaded = files.upload()
csv_filename = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[csv_filename]))

# Suppression des colonnes non utilisées pour éviter des fuites de données
for col in ['split', 'quality']:
    if col in df.columns:
        df = df.drop(columns=[col])
        print(f"Colonne '{col}' supprimée.")

print(f'CSV chargé : {len(df)} lignes')
print('Colonnes :', list(df.columns))
print(df['label'].value_counts())


Uploadez rsna_samples.csv


Saving rsna_samples.csv to rsna_samples.csv
Colonne 'split' supprimée.
Colonne 'quality' supprimée.
CSV chargé : 30227 lignes
Colonnes : ['case_id', 'image_path', 'label']
label
uncertain            11821
suspected_opacity     9555
normal                8851
Name: count, dtype: int64


In [ ]:
# ═══════════════════════════════
# STEP 3. Authentification Kaggle
# ═══════════════════════════════
# Uploadez votre fichier kaggle.json (disponible sur https://www.kaggle.com/settings > API Tokens > Create Legacy API Key)

from google.colab import files as colab_files

print('Uploadez votre kaggle.json')
uploaded_kaggle = colab_files.upload()
kaggle_filename = list(uploaded_kaggle.keys())[0]

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'wb') as f:
    f.write(uploaded_kaggle[kaggle_filename])
os.chmod('/root/.kaggle/kaggle.json', 0o600) # Sécurité: lecture owner seulement
print('kaggle.json installé et sécurisé.')


Uploadez votre kaggle.json


Saving kaggle.json to kaggle.json
kaggle.json installé et sécurisé.


In [23]:
# ════════════════════════════════════════════════════
# STEP 4. Téléchargement des images RSNA depuis Kaggle
# ════════════════════════════════════════════════════
# Dataset : rsna-pneumonia-processed-dataset
# Téléchargé dans /content/rsna/ et décompressé directement

!pip install -q kaggle --upgrade
!kaggle datasets download -d iamtapendu/rsna-pneumonia-processed-dataset -p /content/rsna --unzip -q

from pathlib import Path
from collections import Counter

rsna_root = Path('/content/rsna')
all_pngs  = list(rsna_root.rglob('*.png'))

folder_counts = Counter(str(p.parent) for p in all_pngs)
print(f'{len(all_pngs)} images PNG trouvées sur disque')
print('Répartition par dossier :')
for folder, count in sorted(folder_counts.items()):
    print(f'  {count:6d}  {folder}')

# Index global case_id → Path
# png_index permet de retrouver n'importe quel case_id sans supposer dans quel sous-dossier il se trouve
# En cas de doublon de nom entre dossiers, le dernier rencontré gagne
png_index: dict = {}
for p in all_pngs:
    if p.stem in png_index:
        print(f' case_id en double sur disque: {p.stem}')
        print(f' {png_index[p.stem]}  ←  ignoré')
        print(f' {p}  ←  retenu')
    png_index[p.stem] = p

print(f'\n{len(png_index)} case_id uniques indexés (variable png_index)')

# Réconciliation avec le CSV
csv_ids   = set(df['case_id'].astype(str))
found     = csv_ids & set(png_index.keys())
not_found = csv_ids - set(png_index.keys())
print(f'\nCSV       : {len(csv_ids)} case_id uniques')
print(f'Trouvés   : {len(found)}')
print(f'Manquants : {len(not_found)}')
if not_found:
    print(f'Exemples: {sorted(not_found)[:5]}')
    print('Ces images ne sont pas dans le dataset Kaggle téléchargé. Seuls les cas trouvés seront utilisés.')


Dataset URL: https://www.kaggle.com/datasets/iamtapendu/rsna-pneumonia-processed-dataset
License(s): apache-2.0
11556 images PNG trouvées sur disque
Répartition par dossier :
    3000  /content/rsna/Test
    8556  /content/rsna/Training/Images

11556 case_id uniques indexés (variable png_index)

CSV       : 26684 case_id uniques
Trouvés   : 8556
Manquants : 18128
Exemples: ['626adddd-9800-4304-b1f6-1444971946af', '626eb2ae-ca78-43fc-bbbc-5cf95bb328bb', '626f28a9-ec6f-4e44-b393-503ab394358d', '627567ae-907d-4d83-907d-79db88f48303', '627573e3-954c-41fb-959c-703f2ed7e92f']
Ces images ne sont pas dans le dataset Kaggle téléchargé. Seuls les cas trouvés seront utilisés.


In [24]:
# ═════════════════════════════════════════════════════════════════════
# STEP 5. Résolution des chemins images + split train/val reproductible
# ═════════════════════════════════════════════════════════════════════
#  1. Détecte automatiquement dossier contenant les PNG
#  2. Déduplique CSV sur (case_id, label)
#  3. Filtre cas dont l'image est absente sur disque
#  4. Construit un split train/val STRATIFIÉ et ÉQUILIBRÉ par classe, avec random.seed(SEED) pour reproductibilité

import random
from pathlib import Path
from collections import Counter

# 1. Détection du dossier images
# Chemin attendu après décompression Kaggle
rsna_img_dir = Path('/content/rsna/Training/Images')
if not rsna_img_dir.exists():
    # Fallback: cherche le dossier parent des premiers PNG trouvés
    all_imgs = list(Path('/content/rsna').rglob('*.png'))
    rsna_img_dir = all_imgs[0].parent if all_imgs else Path('/content/rsna')
    print('Dossier images détecté automatiquement :', rsna_img_dir)
else:
    print('Dossier images :', rsna_img_dir)

# 2. Déduplication
# Supprime doublons (même case_id + label) qui biaisent entraînement
df_clean = df.drop_duplicates(subset=['case_id', 'label']).copy()
print(f'Après déduplication : {len(df_clean)} lignes (was {len(df)})')

# 3. Filtrage images manquantes
cases = []
missing = 0
for _, row in df_clean.iterrows():
    img_path = rsna_img_dir / f'{row["case_id"]}.png'
    if img_path.exists():
        cases.append({
            'case_id': row['case_id'],
            'label': row['label'],
            'image_path': str(img_path),
        })
    else:
        missing += 1

print(f'{len(cases)} cas valides, {missing} images manquantes')
print('Distribution :', Counter(c["label"] for c in cases))

# 4. Paramètres du split (peut être ajuster selon votre GPU)
N_PER_CLASS_TRAIN = 55
N_PER_CLASS_VAL   = 15

ALL_LABELS = ['normal', 'suspected_opacity', 'uncertain']

# Garantit un split identique à chaque exécution du notebook
random.seed(SEED)
train_cases, val_cases = [], []
for label in ALL_LABELS:
    subset = [c for c in cases if c['label'] == label]
    if len(subset) == 0:
        print(f'  ⚠️  Classe "{label}" absente du dataset.')
        continue
    random.shuffle(subset)
    n_train = min(N_PER_CLASS_TRAIN, max(0, len(subset) - N_PER_CLASS_VAL))
    n_val   = min(N_PER_CLASS_VAL,   len(subset) - n_train)
    train_cases.extend(subset[:n_train])
    val_cases.extend(subset[n_train:n_train + n_val])
    print(f'  {label:20s} → train={n_train}, val={n_val}  (dispo={len(subset)})')

# Remélange listes avec même seed pour éviter tout ordre résiduel
random.seed(SEED)
random.shuffle(train_cases)
random.shuffle(val_cases)

print(f'\nTrain total : {len(train_cases)} | Val total : {len(val_cases)}')
print('Train dist :', Counter(c["label"] for c in train_cases))
print('Val dist   :', Counter(c["label"] for c in val_cases))


Dossier images : /content/rsna/Training/Images
Après déduplication : 26684 lignes (was 30227)
8556 cas valides, 18128 images manquantes
Distribution : Counter({'uncertain': 3589, 'normal': 2756, 'suspected_opacity': 2211})
  normal               → train=55, val=15  (dispo=2756)
  suspected_opacity    → train=55, val=15  (dispo=2211)
  uncertain            → train=55, val=15  (dispo=3589)

Train total : 165 | Val total : 45
Train dist : Counter({'suspected_opacity': 55, 'uncertain': 55, 'normal': 55})
Val dist   : Counter({'suspected_opacity': 15, 'normal': 15, 'uncertain': 15})


In [25]:
# ════════════════════════════════════════
# STEP 6. Authentification HuggingFace Hub
# ════════════════════════════════════════
# Nécessaire pour télécharger google/medgemma-4b-pt (modèle à accès restreint)
# Obtenez un token sur : https://huggingface.co/settings/tokens
# Vérifiez également que vous avez accepté les conditions d'utilisation sur : https://huggingface.co/google/medgemma-4b-pt

from huggingface_hub import login
login()  # Ouvre une boîte de saisie du token dans Colab

In [26]:
# ══════════════════════════════════════════════════
# STEP 7. Chargement MedGemma 4B-PT en 4-bit (QLoRA)
# ══════════════════════════════════════════════════

import warnings
import torch
from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

warnings.filterwarnings(
    'ignore',
    message='_check_is_size',
    category=FutureWarning,
)

MODEL_ID = 'google/medgemma-4b-pt'

# Configuration quantification 4-bit NF4
# bnb_4bit_use_double_quant : double quantification des constantes de quantification → économise ~0.4 bit/param supplémentaire
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4', #  modèle est chargé en 4-bit NF4, optimal pour poids LLM ce qui réduit VRAM de ~16 GB (bf16) à ~5-6 GB
    bnb_4bit_compute_dtype=torch.bfloat16, # Calculs en bf16 même si stockage 4-bit
    bnb_4bit_use_double_quant=True,
)

# Chargement du processor
processor = AutoProcessor.from_pretrained(MODEL_ID, backend='torchvision')

# Chargement du modèle en 4-bit
# device_map='auto' répartit automatiquement Couches sur GPU/CPU selon VRAM disponible, utile si le modèle dépasse la VRAM GPU seule.
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.bfloat16,
)

# Préparation pour entraînement k-bit
# (active gradient checkpointing, cast couches normalization en fp32 pour stabilité et désactive cache de la couche embedding)
model = prepare_model_for_kbit_training(model)

# Configuration LoRA
# r=16: rang des matrices LoRA, compromis choisi à cause de VRAM
# lora_alpha=32: facteur de mise à l'échelle (alpha/r = 2 : amplification ×2)
# target_modules: projections d'attention (q/k/v/o) + couches FFN (gate/up/down)
# lora_dropout=0.05: régularisation légère pour éviter l'overfitting
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()  # ~1-2% des params totaux sont entraînables
print('Device:', next(model.parameters()).device)


processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.57k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/133 [00:00<?, ?B/s]

trainable params: 32,788,480 || all params: 4,332,867,952 || trainable%: 0.7567
Device: cuda:0


In [27]:
# ═══════════════════════════════
# STEP 8. Construction du dataset
# ═══════════════════════════════
# Chaque exemple est formaté en conversation Gemma (ChatML-like):
#   <start_of_turn>system → instructions du système
#   <start_of_turn>user → token image + prompt de classification
#   <start_of_turn>model → réponse JSON cible (supervisée)

from datasets import Dataset
from PIL import Image
import json

# Token de début d'image
BOI = processor.boi_token

# Prompts système et utilisateur
SYSTEM_PROMPT = (
    'You are an educational radiology assistant for engineering students. '
    'You are not a clinician. Analyze the frontal chest X-ray and return '
    'only the classification label as valid JSON.'
)

USER_PROMPT = (
    'Classify this frontal chest X-ray. '
    'Return ONLY valid JSON with exactly these fields:\n'
    "{\n"
    '  "image_quality": "good | limited | poor",\n'
    '  "predicted_class": "normal | suspected_opacity | uncertain",\n'
    '  "confidence": 0.0,\n'
    '  "visual_evidence": ["short factual observation"],\n'
    '  "justification": "2 to 4 cautious sentences strictly based on visible evidence",\n'
    '  "limitations": ["image quality", "projection", "lack of clinical context"],\n'
    '  "warning": "Educational prototype only. Not for diagnosis. A qualified clinician must verify the image."\n'
    "}\n"
    'Rules:\n'
    '- Use "suspected_opacity" when there is a visible pulmonary opacity or consolidation.\n'
    '- Use "normal" when no significant abnormality is visible.\n'
    '- Use "uncertain" when image quality is poor, the view or findings are ambiguous.\n'
    '- If JSON cannot be guaranteed, return "uncertain"'
)

# Confidence cible par classe
# 'uncertain' reçoit intentionnellement 0.60 pour refléter l'ambiguïté.
LABEL_CONFIDENCE = {
    'normal':            0.95,
    'suspected_opacity': 0.90,
    'uncertain':         0.60,
}

def build_conversation(case: dict) -> str:
    """Formate un cas en texte de conversation Gemma multi-tour.

    Format :
        <start_of_turn>system\n...\n<end_of_turn>\n
        <start_of_turn>user\n<BOI>\n...\n<end_of_turn>\n
        <start_of_turn>model\n{JSON}\n<end_of_turn>
    """
    answer = json.dumps({
        'predicted_class': case['label'],
        'confidence': LABEL_CONFIDENCE.get(case['label'], 0.80),
    })
    text = (
        f'<start_of_turn>system\n{SYSTEM_PROMPT}<end_of_turn>\n'
        f'<start_of_turn>user\n{BOI}\n{USER_PROMPT}<end_of_turn>\n'
        f'<start_of_turn>model\n{answer}<end_of_turn>'
    )
    return text

def preprocess(cases_list: list) -> dict:
    """Construit les listes parallèles texts/images à partir des cas."""
    texts, images = [], []
    for case in cases_list:
        texts.append(build_conversation(case))
        # Conversion RGB obligatoire:PNG RSNA sont en niveaux de gris
        images.append(Image.open(case['image_path']).convert('RGB'))
    return {'text': texts, 'image': images}

print('Construction du dataset train...')
train_data = preprocess(train_cases)
print('Construction du dataset val...')
val_data = preprocess(val_cases)

train_dataset = Dataset.from_dict({'text': train_data['text'], 'image': train_data['image']})
val_dataset = Dataset.from_dict({'text': val_data['text'],   'image': val_data['image']})

print(f'Train dataset : {len(train_dataset)} exemples')
print(f'Val dataset   : {len(val_dataset)} exemples')

# Affichage d'un exemple formaté
print('\nExemple texte (1er cas train) :')
print(train_dataset[0]['text'])


Construction du dataset train...
Construction du dataset val...
Train dataset : 165 exemples
Val dataset   : 45 exemples

Exemple texte (1er cas train) :
<start_of_turn>system
You are an educational radiology assistant for engineering students. You are not a clinician. Analyze the frontal chest X-ray and return only the classification label as valid JSON.<end_of_turn>
<start_of_turn>user
<start_of_image>
Classify this frontal chest X-ray. Return ONLY valid JSON with exactly these fields:
{
  "image_quality": "good | limited | poor",
  "predicted_class": "normal | suspected_opacity | uncertain",
  "confidence": 0.0,
  "visual_evidence": ["short factual observation"],
  "justification": "2 to 4 cautious sentences strictly based on visible evidence",
  "limitations": ["image quality", "projection", "lack of clinical context"],
  "warning": "Educational prototype only. Not for diagnosis. A qualified clinician must verify the image."
}
Rules:
- Use "suspected_opacity" when there is a visibl

In [28]:
# ════════════════════════════════════════════════════════════
# STEP 9. Collator multimodal personnalisé (MedGemma / Gemma3)
# ════════════════════════════════════════════════════════════
# Le SFTTrainer de trl attend un data_collator qui retourne un dict de tenseurs PyTorch.
# Ce collator :
#   1. Encode texte + image pour chaque exemple du batch
#   2. Padde manuellement à gauche à la longueur max du batch (pad_id)
#   3. Construit les labels en masquant (-100):
#        - les tokens de padding,
#        - tous les tokens du prompt (system + user) jusqu'au tour 'model',
#        → la loss n'est calculée QUE sur la réponse JSON cible
#   4. Gère optionnellement token_type_ids si le processor les produit

import torch
from dataclasses import dataclass
from typing import Any, Dict, List

@dataclass
class MedGemmaCollator:
    """Collator multimodal pour MedGemma/Gemma3 avec masquage du prompt.

    Args:
        processor  : AutoProcessor chargé en Cellule 7.
        max_length : longueur max de tokenisation (tronque si dépassé).
    """
    processor: Any
    max_length: int = 512

    def __call__(self, features: List[Dict]) -> Dict[str, torch.Tensor]:
        texts  = [f['text']  for f in features]
        images = [f['image'] for f in features]

        # Encodage individuel
        # encode chaque exemple séparément car images peuvent avoir dimensions différentes avant resize du processor
        encoded_list = []
        for text, image in zip(texts, images):
            enc = self.processor(
                text=text,
                images=[image],
                return_tensors='pt',
                truncation=True,
                max_length=self.max_length,
            )
            encoded_list.append(enc)

        # Padding manuel (gauche)
        # génération auto-regressive lit de droite, donc les tokens utiles doivent être à droite du padding
        max_len = max(e['input_ids'].shape[-1] for e in encoded_list)
        pad_id  = self.processor.tokenizer.pad_token_id

        input_ids_list, attention_mask_list = [], []
        pixel_values_list, labels_list = [], []

        for enc in encoded_list:
            seq_len = enc['input_ids'].shape[-1]
            pad_len = max_len - seq_len

            ids  = torch.nn.functional.pad(enc['input_ids'][0],      (pad_len, 0), value=pad_id)
            mask = torch.nn.functional.pad(enc['attention_mask'][0], (pad_len, 0), value=0)

            # Masquage des labels
            # -100 est la valeur ignorée par CrossEntropyLoss (convention PyTorch)
            # On masque : (a) le padding, (b) tous les tokens avant la réponse
            lbl = ids.clone()
            lbl[:pad_len]       = -100 # (a) tokens de padding
            lbl[lbl == pad_id]  = -100 # (a) pad_id résiduel

            # (b) Recherche du marqueur de début de réponse : '<start_of_turn>model\n'
            # Tout ce qui précède (system + user) est masqué
            model_turn_ids = self.processor.tokenizer.encode(
                '<start_of_turn>model\n', add_special_tokens=False
            )
            mt_len    = len(model_turn_ids)
            mt_tensor = torch.tensor(model_turn_ids)
            for j in range(len(ids) - mt_len):
                if (ids[j:j + mt_len] == mt_tensor).all():
                    lbl[:j + mt_len] = -100 # masque prompt + marqueur
                    break

            input_ids_list.append(ids)
            attention_mask_list.append(mask)
            pixel_values_list.append(enc['pixel_values']) # shape (1, C, H, W)
            labels_list.append(lbl)

        # Assemblage du batch
        batch = {
            'input_ids': torch.stack(input_ids_list),
            'attention_mask': torch.stack(attention_mask_list),
            'pixel_values': torch.cat(pixel_values_list, dim=0), # (B, C, H, W)
            'labels': torch.stack(labels_list),
        }

        # Certains checkpoints Gemma3 produisent token_type_ids, on les padde si présents
        if 'token_type_ids' in encoded_list[0]:
            ttids = [
                torch.nn.functional.pad(
                    e['token_type_ids'][0],
                    (max_len - e['token_type_ids'].shape[-1], 0),
                    value=0
                ) for e in encoded_list
            ]
            batch['token_type_ids'] = torch.stack(ttids)

        return batch

collator = MedGemmaCollator(processor=processor)
print('Collator prêt')


Collator prêt


In [29]:
# ═══════════════════════════════════════════════════
# STEP 10. Configuration des arguments d'entraînement
# ═══════════════════════════════════════════════════

from transformers import TrainingArguments
from trl import SFTTrainer

OUTPUT_DIR = '/content/medgemma_rsna_qlora'

# Détection automatique du dtype optimal
# bf16 (bfloat16) est plus stable que fp16 pour les LLMs mais nécessite un GPU Ampere ou plus récent (A100, L4). T4 → fp16
use_bf16 = torch.cuda.is_bf16_supported()
use_fp16 = not use_bf16
print(f'Mode précision : {"bf16" if use_bf16 else "fp16"}')

# Calcul dynamique de warmup_steps
# on calcule warmup_steps explicitement (5% des steps totaux)
steps_per_epoch = len(train_dataset) // (1 * 8) # batch_size=1, grad_accum=8
total_steps = steps_per_epoch * 3 # 3 epochs
warmup_steps = max(1, int(total_steps * 0.05))
print(f'Steps/epoch={steps_per_epoch}, total={total_steps}, warmup={warmup_steps}')

# TrainingArguments documente tous les hyperparamètres
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=1,       # Batch 1 pour tenir en VRAM sur T4
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,       # Batch effectif = 8 (simule batch_size=8)
    learning_rate=2e-4,                  # LR usuel pour QLoRA
    lr_scheduler_type='cosine',          # Décroissance cosinus (stable pour LLMs)
    warmup_steps=warmup_steps,           # Montée progressive du LR au début
    optim='paged_adamw_8bit',            # AdamW paginé 8-bit : économise ~2 GB VRAM
    fp16=use_fp16,
    bf16=use_bf16,
    logging_steps=10,                    # Log la loss tous les 10 steps
    eval_strategy='epoch',               # Évaluation à la fin de chaque epoch
    save_strategy='epoch',               # Sauvegarde checkpoint chaque epoch
    load_best_model_at_end=True,         # Recharge le meilleur checkpoint en fin
    metric_for_best_model='eval_loss',
    report_to='none',                    # Désactive wandb/tensorboard
    dataloader_num_workers=0,            # 0 = déterministe (pas de fork)
    remove_unused_columns=False,         # Nécessaire pour le collator custom
    gradient_checkpointing=True,         # Recalcul des activations → -30% VRAM
    gradient_checkpointing_kwargs={'use_reentrant': False}, # Requis avec PEFT
    max_grad_norm=0.3,                   # Gradient clipping : stabilité training
    seed=SEED,                           # Seed Trainer (shuffling DataLoader, etc.)
    data_seed=SEED,                      # Seed spécifique au DataLoader
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collator
)

print('Trainer configuré')
print(f'Steps effectifs/epoch : {steps_per_epoch}')


Mode précision : bf16
Steps/epoch=20, total=60, warmup=3
Trainer configuré
Steps effectifs/epoch : 20


In [ ]:
# ══════════════════════════
# STEP 11. Fine-tuning QLoRA
# ══════════════════════════
# Cette cellule lance l'entraînement. Avant de démarrer :
#   1. Vide le cache GPU pour libérer la VRAM fragmentée
#   2. Affiche la VRAM libre pour vérifier qu'on a assez de marge
#   3. Supprime les warnings de dépréciation du tokenizer (tokens PAD/BOS/EOS) qui sont inoffensifs mais polluent les logs d'entraînement

import gc
import warnings

# Alignement config modèle ↔ tokenizer
# Le tokenizer MedGemma peut avoir des tokens spéciaux différents de ceux enregistrés dans model.config / model.generation_config
# Transformers détecte cet écart et le corrige tout seul (avec un warning). On le fait explicitement ici pour éviter le message
tok = processor.tokenizer
for config_obj in [model.config, getattr(model, 'generation_config', None)]:
    if config_obj is None:
        continue
    if hasattr(config_obj, 'pad_token_id') and tok.pad_token_id is not None:
        config_obj.pad_token_id = tok.pad_token_id
    if hasattr(config_obj, 'bos_token_id') and tok.bos_token_id is not None:
        config_obj.bos_token_id = tok.bos_token_id
    if hasattr(config_obj, 'eos_token_id') and tok.eos_token_id is not None:
        config_obj.eos_token_id = tok.eos_token_id

# Supprime  warnings résiduels de tokens spéciaux pendant entraînement
warnings.filterwarnings('ignore', message='.*PAD/BOS/EOS tokens.*')
warnings.filterwarnings('ignore', message='.*image_token.*boi_token.*')

# Nettoyage mémoire GPU
gc.collect()
torch.cuda.empty_cache()
print(f'VRAM libre avant training : {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')

# Lancement de l'entraînement
# Durée estimée: ~2h sur T4 pour N_PER_CLASS_TRAIN = 55, N_PER_CLASS_VAL = 15
print('Démarrage du fine-tuning QLoRA...')
trainer.train()
print('Fine-tuning terminé !')


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


VRAM libre avant training : 10.8 GB
Démarrage du fine-tuning QLoRA...


[transformers] Deprecated: `processor.image_token` will switch from returning `tokenizer.image_token` to `tokenizer.boi_token` in v5.11.


Epoch,Training Loss,Validation Loss


In [ ]:
# ══════════════════════════════════════════════════════════
# STEP 12. Sauvegarde et téléchargement des adaptateurs LoRA
# ══════════════════════════════════════════════════════════
# Note : LORA_SAVE_PATH a déjà été rempli en Cellule 12. Cette cellule est un point de sauvegarde explicite/re-téléchargement

import shutil
from pathlib import Path
from google.colab import files

LORA_SAVE_PATH = '/content/medgemma_rsna_lora_adapters'

def save_lora():
    # sauvegarde
    model.save_pretrained(LORA_SAVE_PATH)
    processor.save_pretrained(LORA_SAVE_PATH)
    print('Adaptateurs LoRA sauvegardés dans', LORA_SAVE_PATH)

    # vérification contenu dossier
    path = Path(LORA_SAVE_PATH)
    if not path.exists() or len(list(path.iterdir())) == 0:
        print("⚠️ Dossier vide ou inexistant → relance sauvegarde...")
        # relance une 2e fois
        model.save_pretrained(LORA_SAVE_PATH)
        processor.save_pretrained(LORA_SAVE_PATH)

    # Re-check après retry
    files_list = list(path.iterdir())

    if len(files_list) == 0:
        raise RuntimeError("Échec: le dossier LoRA est toujours vide après retry")

save_lora()

# Liste et taille des fichiers sauvegardés
print('Fichiers :')
for p in sorted(Path(LORA_SAVE_PATH).iterdir()):
    size_mb = p.stat().st_size / 1e6
    print(f'  {p.name:40s} {size_mb:.1f} MB')

# Compression et téléchargement dans Colab
shutil.make_archive('/content/lora_adapters', 'zip', LORA_SAVE_PATH)
files.download('/content/lora_adapters.zip')

print('Téléchargement lancé.')

In [ ]:
# ═════════════════════════════════════════════════════════════════
# STEP 13. Merge des adaptateurs LoRA + évaluation sur le split val
# ═════════════════════════════════════════════════════════════════
#   1. Libère le modèle QLoRA de la VRAM
#   2. Recharge le modèle de base en bf16/fp16 et fusionne les poids LoRA
#   3. Évalue sur val_cases avec décodage JSON et fallback textuel
#   4. Affiche accuracy, macro F1 et rapport de classification

import re, json, gc
from PIL import Image
from sklearn.metrics import accuracy_score, f1_score, classification_report
from peft import PeftModel
from transformers import AutoModelForImageTextToText

LORA_SAVE_PATH = '/content/medgemma_rsna_lora_adapters'

# 1. Libération VRAM
# pour éviter Out Of Memory GPU
del model
gc.collect()
torch.cuda.empty_cache()
print(f'VRAM libre après del : {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')

# 2. Rechargement base + merge LoRA
# charge en bf16 (ou fp16 sur T4) pour maximiser précision en inférence
# merge_and_unload() fusionne poids LoRA dans modèle de base et supprime couches PEFT
use_bf16 = torch.cuda.is_bf16_supported()
dtype = torch.bfloat16 if use_bf16 else torch.float16
print(f'Dtype inférence : {dtype}')

base = AutoModelForImageTextToText.from_pretrained(
    'google/medgemma-4b-pt',
    torch_dtype=dtype,
    device_map='auto',
)
merged_model = PeftModel.from_pretrained(base, LORA_SAVE_PATH)
merged_model = merged_model.merge_and_unload()
merged_model.eval()
merged_model.config.use_cache = True  # Active le KV-cache pour l'inférence
print('Modèle mergé prêt')

# 3. Prompt d'évaluation
# EVAL_PROMPT: texte passé au processor en inférence (pas de réponse)
# Il doit correspondre exactement au format vu pendant le training
EVAL_PROMPT = (
    f'<start_of_turn>system\n{SYSTEM_PROMPT}<end_of_turn>\n'
    f'<start_of_turn>user\n{BOI}\n{USER_PROMPT}<end_of_turn>\n'
    f'<start_of_turn>model\n'
)

# Clés à passer à generate() (on exclut les champs non-reconnus)
GENERATE_KEYS = {'input_ids', 'attention_mask', 'pixel_values'}
VALID_LABELS  = {'normal', 'suspected_opacity', 'uncertain'}

def evaluate_case(image_path: str) -> tuple:
    """Prédit la classe d'un cas radiologique.

    Retourne (predicted_class, raw_text) avec un fallback textuel si le
    JSON est malformé ou si predicted_class n'est pas dans VALID_LABELS.
    """
    image = Image.open(image_path).convert('RGB')
    inputs = processor(text=EVAL_PROMPT, images=[image], return_tensors='pt')
    inputs = {k: v.to(merged_model.device) for k, v in inputs.items() if k in GENERATE_KEYS}

    with torch.no_grad():
        outputs = merged_model.generate(
            **inputs,
            max_new_tokens=64, # Suffisant pour {"predicted_class":...,"confidence":...}
            do_sample=False, # Greedy decoding → déterministe
            use_cache=True,
        )

    # Décode uniquement les tokens générés
    input_len = inputs['input_ids'].shape[-1]
    text = processor.decode(outputs[0][input_len:], skip_special_tokens=True)

    try:
        # Extraction du premier objet JSON dans la réponse
        match = re.search(r'\{.*?\}', text, re.DOTALL)
        result = json.loads(match.group()) if match else {}
        pred = result.get('predicted_class', 'uncertain')
        if pred not in VALID_LABELS:
            pred = 'uncertain'
    except Exception:
        # Fallback: recherche textuelle des labels dans la réponse brute
        text_l = text.lower()
        if 'suspected_opacity' in text_l or 'opacity' in text_l:
            pred = 'suspected_opacity'
        elif 'normal' in text_l:
            pred = 'normal'
        else:
            pred = 'uncertain'

    return pred, text

# 4. Boucle d'évaluation
y_true, y_pred = [], []
for i, case in enumerate(val_cases):
    pred, raw = evaluate_case(case['image_path'])
    y_true.append(case['label'])
    y_pred.append(pred)
    ok = '✓' if pred == case['label'] else '✗'
    print(f'[{i+1:03d}/{len(val_cases)}] {ok} label={case["label"]:20s} pred={pred}')

print('\n=== Résultats validation ===')
print(f'Accuracy : {accuracy_score(y_true, y_pred):.4f}')
print(f'Macro F1 : {f1_score(y_true, y_pred, average="macro", zero_division=0):.4f}')
print('\nRapport détaillé :')
print(classification_report(
    y_true, y_pred,
    labels=['normal', 'suspected_opacity', 'uncertain'],
    zero_division=0
))


In [ ]:
# ═════════════════════════════════════════════════
# STEP 14 (optionnel) - Merge LoRA → modèle complet
# ═════════════════════════════════════════════════
# pour déployer le modèle sans PEFT (inférence standard)
# /!\ Nécessite ~16 GB de VRAM supplémentaire, à lancer sur A100 uniquement
# Le modèle mergé pèse ~8 GB, utilisez push_to_hub() pour l'uploader sur HuggingFace Hub plutôt que de le télécharger localement

from peft import PeftModel
from transformers import AutoModelForImageTextToText

print('Chargement modèle base en bf16 pour merge...')
base_model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map='auto',
)

# Fusion des poids LoRA dans le modèle de base
# merge_and_unload(): fusionne, puis supprime les couches PEFT → modèle standard
merged_model = PeftModel.from_pretrained(base_model, LORA_SAVE_PATH)
merged_model = merged_model.merge_and_unload()

MERGED_PATH = '/content/medgemma_rsna_merged'
merged_model.save_pretrained(MERGED_PATH, safe_serialization=True) # Format safetensors
processor.save_pretrained(MERGED_PATH)
print('Modèle mergé sauvegardé dans', MERGED_PATH)
print('(~8 GB — utilisez HuggingFace Hub push_to_hub() pour l\'uploader)')


## Notes techniques

**Modèle :** `google/medgemma-4b-pt` (base pré-entraîné, vision-language)

**QLoRA :** quantification 4-bit NF4 + adaptateurs LoRA `r=16, alpha=32` sur les couches d'attention et FFN

**Loss :** calculée uniquement sur les tokens de réponse (masquage du prompt utilisateur)

**GPU recommandé :**
- T4 (15 GB) : `N_PER_CLASS_TRAIN=100`, `batch=1`, `grad_accum=8`
- L4 (24 GB) : `N_PER_CLASS_TRAIN=150`, `batch=2`, `grad_accum=4`
- A100 (40/80 GB) : `N_PER_CLASS_TRAIN=200+`, `batch=4`, `grad_accum=2`

**Format de sortie entraîné :**
```json
{
  "image_quality": "good | limited | poor",
  "predicted_class": "normal | suspected_opacity | uncertain",
  "confidence": 0.0,
  "visual_evidence": ["short factual observation"],
  "justification": "2 to 4 cautious sentences based only on visible evidence",
  "limitations": ["image quality", "projection", "lack of clinical context"],
  "warning": "Educational prototype only. Not for diagnosis. A qualified clinician must verify the image."
}
```

**Réutiliser les adaptateurs LoRA :**
```python
from peft import PeftModel
model = AutoModelForImageTextToText.from_pretrained('google/medgemma-4b-pt', ...)
model = PeftModel.from_pretrained(model, 'chemin/vers/lora_adapters')
```